In [ ]:
# pip install kagglehub[pandas-datasets]
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

# ----------------------------
# 1) Load from KaggleHub
# ----------------------------
# IMPORTANT:
# - You MUST set file_path to the actual CSV filename inside the Kaggle dataset.
# - Example names are usually like:
#   "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
#   "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
#   etc.

file_path = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"  # <-- change if needed

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "chethuhn/network-intrusion-dataset",
    file_path
)

print("Loaded shape:", df.shape)
print("First 5 records:\n", df.head())

/tmp/ipython-input-355/3775881620.py:19: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 17.8M/17.8M [00:00<00:00, 75.7MB/s]

Extracting zip of Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...


Loaded shape: (225745, 79)
First 5 records:
     Destination Port   Flow Duration   Total Fwd Packets  \
0              54865               3                   2   
1              55054             109                   1   
2              55055              52                   1   
3              46236              34                   1   
4              54863               3                   2   

    Total Backward Packets  Total Length of Fwd Packets  \
0                        0                           12   
1                        1                            6   
2                        1                            6   
3                        1                            6   
4                        0                           12   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                             0                       6   
1                             6                       6   
2                             6                       6   
3  

In [ ]:
# 2) Robust column cleaning + label detection
# ----------------------------
df.columns = df.columns.astype(str).str.strip()

# detect label column safely (CIC often has "Label" or it is the last column)
candidates = [c for c in df.columns if c.lower() in ["label", "class", "target", "attack"]]
label_col = candidates[0] if len(candidates) > 0 else df.columns[-1]

print("\nDetected label column:", label_col)
print("Unique labels (sample):", df[label_col].astype(str).str.strip().unique()[:10])

# Clean data
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Binary label: BENIGN=0, others=1
y = (df[label_col].astype(str).str.strip().str.upper() != "BENIGN").astype(np.int64).values

# Use numeric features only (drop label column)
X = df.drop(columns=[label_col], errors="ignore").select_dtypes(include=[np.number]).values.astype(np.float32)

print("\nFinal X shape:", X.shape, "y shape:", y.shape, "Attack ratio:", y.mean())


Detected label column: Label
Unique labels (sample): ['BENIGN' 'DDoS']

Final X shape: (225711, 78) y shape: (225711,) Attack ratio: 0.5672076239084493


In [ ]:
# 3) Train/Val/Test split + scaling
# ----------------------------
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# Combine for single-graph training (transductive style)
X_all = np.vstack([X_train, X_val, X_test]).astype(np.float32)
y_all = np.concatenate([y_train, y_val, y_test]).astype(np.int64)

n_train = len(X_train)
n_val = len(X_val)
n_test = len(X_test)

train_idx = np.arange(0, n_train)
val_idx = np.arange(n_train, n_train + n_val)
test_idx = np.arange(n_train + n_val, n_train + n_val + n_test)

In [ ]:
# 4) Build kNN Flow-Graph (nodes=flows, edges=similar flows)
# ----------------------------
k = 15  # try 10/15/20
nn_model = NearestNeighbors(n_neighbors=k + 1, metric="cosine")
nn_model.fit(X_all)

dist, nbrs = nn_model.kneighbors(X_all)

src_list, dst_list = [], []
for i in range(nbrs.shape[0]):
    for j in nbrs[i, 1:]:  # skip itself
        # undirected
        src_list.append(i); dst_list.append(j)
        src_list.append(j); dst_list.append(i)

src = np.array(src_list, dtype=np.int64)
dst = np.array(dst_list, dtype=np.int64)

N = X_all.shape[0]
E = src.shape[0]
print("\nGraph nodes:", N, "edges:", E)

# Sparse adjacency A
indices = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
values = torch.ones(E, dtype=torch.float32)
A = torch.sparse_coo_tensor(indices, values, (N, N)).coalesce()

# Add self-loops
self_idx = torch.arange(N, dtype=torch.long)
I = torch.sparse_coo_tensor(
    torch.stack([self_idx, self_idx], dim=0),
    torch.ones(N, dtype=torch.float32),
    (N, N)
).coalesce()
A = (A + I).coalesce()

# Symmetric normalization: A_norm = D^{-1/2} A D^{-1/2}
deg = torch.sparse.sum(A, dim=1).to_dense()
deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)

row, col = A.indices()
val = A.values() * deg_inv_sqrt[row] * deg_inv_sqrt[col]
A_norm = torch.sparse_coo_tensor(torch.stack([row, col], dim=0), val, (N, N)).coalesce()


Graph nodes: 225711 edges: 6771330


In [ ]:
# 5) Pure PyTorch GCN
# ----------------------------
class SparseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, A_norm):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = torch.sparse.mm(A_norm, x)  # message passing
        x = self.lin(x)
        return x

class GCN(nn.Module):
    def __init__(self, in_dim, hidden=64, dropout=0.3):
        super().__init__()
        self.gcn1 = SparseGCNLayer(in_dim, hidden, dropout)
        self.gcn2 = SparseGCNLayer(hidden, 2, dropout)

    def forward(self, x, A_norm):
        x = self.gcn1(x, A_norm)
        x = F.relu(x)
        x = self.gcn2(x, A_norm)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_t = torch.tensor(X_all, dtype=torch.float32).to(device)
y_t = torch.tensor(y_all, dtype=torch.long).to(device)
A_norm = A_norm.to(device)

model = GCN(in_dim=x_t.shape[1], hidden=64, dropout=0.3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Class weights (handles imbalance)
pos = y_all.mean()
w0 = 1.0
w1 = (1.0 - pos) / max(pos, 1e-6)
class_weight = torch.tensor([w0, w1], dtype=torch.float32).to(device)

def evaluate(idx):
    model.eval()
    with torch.no_grad():
        logits = model(x_t, A_norm)
        probs = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = logits.argmax(dim=1).detach().cpu().numpy()

    y_true = y_all[idx]
    y_pred = pred[idx]
    y_prob = probs[idx]
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    return f1, auc, y_true, y_pred, y_prob



In [ ]:
# 6) Train
# ----------------------------
best_val_f1 = -1
best_state = None

for epoch in range(1, 31):
    model.train()
    opt.zero_grad()

    logits = model(x_t, A_norm)
    loss = F.cross_entropy(logits[train_idx], y_t[train_idx], weight=class_weight)

    loss.backward()
    opt.step()

    val_f1, val_auc, _, _, _ = evaluate(val_idx)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        train_f1, train_auc, _, _, _ = evaluate(train_idx)
        print(f"Epoch {epoch:02d} | Loss {loss.item():.4f} | Train F1 {train_f1:.4f} AUC {train_auc:.4f} | Val F1 {val_f1:.4f} AUC {val_auc:.4f}")

# Load best model
model.load_state_dict(best_state)


Epoch 01 | Loss 0.6899 | Train F1 0.3944 AUC 0.6849 | Val F1 0.3879 AUC 0.6761
Epoch 05 | Loss 0.6058 | Train F1 0.7518 AUC 0.9112 | Val F1 0.7486 AUC 0.9069
Epoch 10 | Loss 0.5206 | Train F1 0.7795 AUC 0.9693 | Val F1 0.7771 AUC 0.9680
Epoch 15 | Loss 0.4485 | Train F1 0.7801 AUC 0.9871 | Val F1 0.7780 AUC 0.9868
Epoch 20 | Loss 0.3855 | Train F1 0.8049 AUC 0.9924 | Val F1 0.8019 AUC 0.9923
Epoch 25 | Loss 0.3319 | Train F1 0.8606 AUC 0.9946 | Val F1 0.8600 AUC 0.9944
Epoch 30 | Loss 0.2851 | Train F1 0.9165 AUC 0.9961 | Val F1 0.9146 AUC 0.9958


<All keys matched successfully>

In [ ]:
# 7) Test
# ----------------------------
test_f1, test_auc, y_true, y_pred, y_prob = evaluate(test_idx)

print("\n=== TEST RESULTS ===")
print("Test F1:", round(test_f1, 4), "| Test AUC:", round(test_auc, 4))
print(classification_report(y_true, y_pred, target_names=["BENIGN", "ATTACK"]))


=== TEST RESULTS ===
Test F1: 0.9147 | Test AUC: 0.996
              precision    recall  f1-score   support

      BENIGN       0.83      0.99      0.91     14653
      ATTACK       0.99      0.85      0.91     19204

    accuracy                           0.91     33857
   macro avg       0.91      0.92      0.91     33857
weighted avg       0.92      0.91      0.91     33857

